In [1]:
import json

# Завантаження своїх прикладів
with open("examples.json", "r", encoding="utf-8") as f:
    exs = json.load(f)

trainset = []

for ex in exs:
    ex_text = ex["text"]
    ex_entities = {"entities" : [tuple(ents) for ents in ex["entities"]]}
    trainset.append((ex_text, ex_entities))

trainset

[('Could you help me to find an apartment, please?',
  {'entities': [(29, 38, 'Service')]}),
 ("Yes please, I'd like to rent a house.",
  {'entities': [(24, 36, 'Service')]}),
 ('Can you please find me a two-bedroom apartment?',
  {'entities': [(25, 46, 'Service')]}),
 ('I need to schedule a visit to the property.',
  {'entities': [(10, 26, 'Service')]}),
 ('Any other condos available in this area?',
  {'entities': [(4, 16, 'Service')]})]

In [2]:
import spacy
from spacy.training.example import Example
from spacy.util import minibatch
import random

nlp = spacy.load("en_core_web_sm")

ner = nlp.get_pipe("ner")
ner.add_label("Service")

examples = []
for text, ann in trainset:
    doc = nlp.make_doc(text)
    example = Example.from_dict(doc, ann)
    examples.append(example)

optimizer = nlp.resume_training()
epochs = 20

# Тренування моделі
for i in range(epochs):
    random.shuffle(examples)
    batches = minibatch(examples, size=2)
    for batch in batches:
        nlp.update(batch)

nlp.to_disk("service_model")

In [3]:
nlp = spacy.load("service_model")

texts = [
    "I want you to find me an apartment in Kyiv with two bedrooms.",
    "If there are other houses available in this area, can you show me a two-bedroom apartment?",
    "I want to buy smartphone.",
    "Find me a picture of car.",
    "I'd like to schedule a visit to the apartment tomorrow."
]

for text in texts:
    doc = nlp(text)
    print("Text:", text)
    for ent in doc.ents:
        if ent.label_ == "Service":
            print(f" - {ent.text} ({ent.label_})")

Text: I want you to find me an apartment in Kyiv with two bedrooms.
 - two bedrooms (Service)
Text: If there are other houses available in this area, can you show me a two-bedroom apartment?
 - two-bedroom apartment (Service)
Text: I want to buy smartphone.
Text: Find me a picture of car.
Text: I'd like to schedule a visit to the apartment tomorrow.
 - schedule a visit (Service)


In [4]:
with open("homes.json", "r", encoding="utf-8") as f:
    dialogues = json.load(f)

training_data = []

for dialogue in dialogues:
    for turn in dialogue.get("turns", []):
        if turn.get("speaker") == "USER":
            utterance = turn.get("utterance", "")
            for frame in turn.get("frames", []):
                intent = frame.get("state", {}).get("active_intent", "")
                if intent and intent != "NONE":
                    training_data.append((utterance, intent))
                    break

training_data[:5]

[('I want apartment where pets are allowed', 'FindApartment'),
 ('one bed room is enough', 'FindApartment'),
 ('Fremont will be fine', 'FindApartment'),
 ('Please find some other apartments', 'FindApartment'),
 ('can you find some other with one baths?', 'FindApartment')]

In [5]:
from spacy.pipeline.textcat import Config

nlp = spacy.blank("en")

config = {
    "threshold": 0.5
}

textcat = nlp.add_pipe("textcat_multilabel", config=config)

labels = set(label for _, label in training_data)
for label in labels:
    textcat.add_label(label)

examples = []
for text, label in training_data:
    doc = nlp.make_doc(text)
    cats = {l: (l == label) for l in labels}
    examples.append(Example.from_dict(doc, {"cats": cats}))

optimizer = nlp.initialize()

for epoch in range(epochs):
    random.shuffle(examples)
    for batch in minibatch(examples, size=8):
        nlp.update(batch, sgd=optimizer)

nlp.to_disk("intent_model")

In [6]:
nlp = spacy.load("intent_model")

test_texts = [
    "I'd like to find an apartment in Fremont.",
    "Can you schedule a visit for me on May 22?",
    "I need a house with two bedrooms and pets allowed.",
    "I'd like to visit this property tomorrow at 10 am.",
    "Are there any other apartments available in this area?"
]

for text in test_texts:
    doc = nlp(text)
    print("Text:", text)
    
    pred_label = ""
    pred_score = -1
    
    for label, score in doc.cats.items():
        if score > pred_score:
            pred_label = label
            pred_score = score
    
    print(f" - Predicted intent: {pred_label} ({pred_score:.2f})")

Text: I'd like to find an apartment in Fremont.
 - Predicted intent: FindApartment (1.00)
Text: Can you schedule a visit for me on May 22?
 - Predicted intent: ScheduleVisit (1.00)
Text: I need a house with two bedrooms and pets allowed.
 - Predicted intent: FindApartment (0.83)
Text: I'd like to visit this property tomorrow at 10 am.
 - Predicted intent: ScheduleVisit (0.99)
Text: Are there any other apartments available in this area?
 - Predicted intent: FindApartment (1.00)
